In [1]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# Resolve legacy IDX paths stored in existing CSVs without rewriting the data.
def _relocated_idx_path(value):
    text = str(value).replace("\\", "/")
    old_repo = "AI-Builders-Hackhaton-2026-Backend/"
    if old_repo in text:
        text = text.split(old_repo, 1)[1]
    old_raw = "data/idx_financial_statements/"
    if text.startswith(old_raw):
        text = "data/idx_financial/raw/" + text[len(old_raw):]
    return Path(text)

from pathlib import Path
import pandas as pd
import re
from collections import Counter, defaultdict

from openpyxl import load_workbook
from tqdm.auto import tqdm

e:\BINUS_CODING\Ai Builders Hackhaton\AI-Builders-Hackhaton-2026-Backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SCAN_FILE = Path(
    "data/idx_financial/inventory/idx_financial_file_scan.csv"
)

OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_label_discovery.csv"
)

CHECKPOINT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_label_discovery_checkpoint.csv"
)

print("Scan file exists:", SCAN_FILE.exists())

Scan file exists: True


In [3]:
scan_df = pd.read_csv(
    SCAN_FILE
)

valid_files_df = (
    scan_df[
        scan_df["scan_status"] == "VALID"
    ]
    .copy()
)

print("Valid XLSX files:", len(valid_files_df))
print("Unique tickers:", valid_files_df["ticker"].nunique())

Valid XLSX files: 15821
Unique tickers: 948


In [4]:
def resolve_file_path(path_text):

    path = _relocated_idx_path(
        str(path_text)
    )

    if path.exists():
        return path

    alternative = Path.cwd() / path

    if alternative.exists():
        return alternative

    return None

In [5]:
def normalize_label(value):

    if value is None:
        return ""

    text = str(value).strip().lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text

In [10]:
SKIP_SHEETS = {
    "context",
    "inlinexbrl",
    "hidden",
    "token",
    "1000000"
}

def should_skip_sheet(sheet_name):

    return (
        str(sheet_name)
        .strip()
        .lower()
        in SKIP_SHEETS
    )

In [11]:
def is_useful_label(value):

    if not isinstance(value, str):
        return False

    text = normalize_label(value)

    if not text:
        return False

    if len(text) < 3:
        return False

    if text.isdigit():
        return False

    return True

In [12]:
def extract_labels_from_file(row):

    file_path = resolve_file_path(
        row["file_path"]
    )

    results = []

    if file_path is None:
        return results

    try:

        workbook = load_workbook(
            _relocated_idx_path(file_path),
            read_only=True,
            data_only=True
        )

        for sheet_name in workbook.sheetnames:

            if should_skip_sheet(
                sheet_name
            ):
                continue

            worksheet = workbook[
                sheet_name
            ]

            for row_number, cells in enumerate(
                worksheet.iter_rows(
                    values_only=True
                ),
                start=1
            ):

                for column_index, value in enumerate(
                    cells
                ):

                    if not is_useful_label(
                        value
                    ):
                        continue

                    label_norm = normalize_label(
                        value
                    )

                    results.append({
                        "ticker": row["ticker"],
                        "year": row["year"],
                        "quarter": row["quarter"],
                        "source_file": row["file_name"],
                        "source_path": row["file_path"],
                        "source_sheet": sheet_name,
                        "row_number": row_number,
                        "column_index": column_index,
                        "source_label": value,
                        "normalized_label": label_norm
                    })

        workbook.close()

    except Exception:
        pass

    return results

In [13]:
test_df = valid_files_df.head(3)

test_results = []

for _, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Testing label discovery",
    unit="file"
):

    test_results.extend(
        extract_labels_from_file(
            row
        )
    )

test_labels_df = pd.DataFrame(
    test_results
)

print("Labels found:", len(test_labels_df))

display(
    test_labels_df.head(100)
)

Testing label discovery: 100%|██████████| 3/3 [00:01<00:00,  1.94file/s]

Labels found: 24732


,ticker,year,quarter,source_file,source_path,source_sheet,row_number,column_index,source_label,normalized_label
0,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,1,0,[1210000] Statement of financial position pres...,[1210000] statement of financial position pres...
1,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,3,0,Laporan posisi keuangan,laporan posisi keuangan
2,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,3,2,Statement of financial position,statement of financial position
3,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,4,1,CurrentYearInstant,currentyearinstant
4,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,4,2,PriorEndYearInstant,priorendyearinstant
...,...,...,...,...,...,...,...,...,...,...
95,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,50,0,Pajak dibayar dimuka lancar,pajak dibayar dimuka lancar
96,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,50,3,Current prepaid taxes,current prepaid taxes
97,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,51,0,Klaim atas pengembalian pajak lancar,klaim atas pengembalian pajak lancar
98,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1210000,51,3,Current claims for tax refund,current claims for tax refund


In [14]:
sheet_summary = (
    test_labels_df
    .groupby(
        [
            "ticker",
            "source_sheet"
        ]
    )
    .size()
    .reset_index(
        name="label_count"
    )
)

display(
    sheet_summary
)

,ticker,source_sheet,label_count
0,ZINC,1210000,543
1,ZINC,1311000,127
2,ZINC,1410000,129
3,ZINC,1410000PY,129
4,ZINC,1510000,357
...,...,...,...
75,ZYRX,1691100,8
76,ZYRX,1692000,1237
77,ZYRX,1693000,1321
78,ZYRX,1693100,9


In [15]:
profit_test_df = (
    test_labels_df[
        test_labels_df[
            "normalized_label"
        ]
        .str.contains(
            r"profit|laba|income",
            regex=True,
            na=False
        )
    ]
    .copy()
)

print(
    "Profit-related labels found:",
    len(profit_test_df)
)

display(
    profit_test_df[
        [
            "ticker",
            "source_sheet",
            "row_number",
            "source_label",
            "normalized_label"
        ]
    ].head(200)
)

Profit-related labels found: 307


,ticker,source_sheet,row_number,source_label,normalized_label
21,ZYRX,1210000,13,Aset keuangan lancar yang diukur pada nilai wa...,aset keuangan lancar yang diukur pada nilai wa...
22,ZYRX,1210000,13,Current financial assets at fair value through...,current financial assets at fair value through...
24,ZYRX,1210000,14,Current financial assets fair value through ot...,current financial assets fair value through ot...
157,ZYRX,1210000,81,Aset keuangan tidak lancar yang diukur pada ni...,aset keuangan tidak lancar yang diukur pada ni...
158,ZYRX,1210000,81,Non-current financial assets at fair value thr...,non-current financial assets at fair value thr...
...,...,...,...,...,...
9169,ZONE,1510000,34,Income taxes refunded (paid) from operating ac...,income taxes refunded (paid) from operating ac...
9175,ZONE,1510000,37,Payments for corporate income tax,payments for corporate income tax
9264,ZONE,1510000,82,Pencairan (penempatan) aset keuangan yang diuk...,pencairan (penempatan) aset keuangan yang diuk...
9265,ZONE,1510000,82,Withdrawal (placement) of financial assets at ...,withdrawal (placement) of financial assets at ...


In [16]:
cashflow_test_df = (
    test_labels_df[
        test_labels_df[
            "normalized_label"
        ]
        .str.contains(
            r"cash flow|arus kas|operating activit|aktivitas operasi",
            regex=True,
            na=False
        )
    ]
    .copy()
)

print(
    "Cash-flow-related labels found:",
    len(cashflow_test_df)
)

display(
    cashflow_test_df[
        [
            "ticker",
            "source_sheet",
            "row_number",
            "source_label",
            "normalized_label"
        ]
    ].head(200)
)

Cash-flow-related labels found: 159


,ticker,source_sheet,row_number,source_label,normalized_label
517,ZYRX,1210000,261,Cadangan lindung nilai arus kas,cadangan lindung nilai arus kas
518,ZYRX,1210000,261,Reserve of cash flow hedges,reserve of cash flow hedges
622,ZYRX,1321000,42,"Keuntungan (kerugian) lindung nilai arus kas, ...","keuntungan (kerugian) lindung nilai arus kas, ..."
623,ZYRX,1321000,42,"Gains (losses) on cash flow hedges, before tax","gains (losses) on cash flow hedges, before tax"
624,ZYRX,1321000,43,Penyesuaian reklasifikasi atas lindung nilai a...,penyesuaian reklasifikasi atas lindung nilai a...
...,...,...,...,...,...
17605,ZINC,1510000,174,Total net cash flows received from (used in) f...,total net cash flows received from (used in) f...
17608,ZINC,1510000,176,"Kas dan setara kas arus kas, awal periode","kas dan setara kas arus kas, awal periode"
17609,ZINC,1510000,176,"Cash and cash equivalents cash flows, beginnin...","cash and cash equivalents cash flows, beginnin..."
17616,ZINC,1510000,180,"Kas dan setara kas arus kas, akhir periode","kas dan setara kas arus kas, akhir periode"


In [17]:
profit_label_summary = (
    profit_test_df
    .groupby(
        "normalized_label"
    )
    .agg(
        occurrences=(
            "normalized_label",
            "size"
        ),
        unique_tickers=(
            "ticker",
            "nunique"
        ),
        example_label=(
            "source_label",
            "first"
        )
    )
    .reset_index()
    .sort_values(
        [
            "unique_tickers",
            "occurrences"
        ],
        ascending=[
            False,
            False
        ]
    )
)

display(
    profit_label_summary.head(200)
)

,normalized_label,occurrences,unique_tickers,example_label
91,reserve for changes in fair value of fair valu...,9,3,Reserve for changes in fair value of fair valu...
93,saldo laba yang belum ditentukan penggunaannya,9,3,Saldo laba yang belum ditentukan penggunaannya
94,saldo laba yang telah ditentukan penggunaannya,9,3,Saldo laba yang telah ditentukan penggunaannya
34,laba (rugi),6,3,Laba (rugi)
52,laporan laba rugi dan penghasilan komprehensif...,6,3,Laporan laba rugi dan penghasilan komprehensif...
...,...,...,...,...
102,tax on other comprehensive income,1,1,Tax on other comprehensive income
106,total other comprehensive income that may be r...,1,1,Total other comprehensive income that may be r...
108,total other comprehensive income that will not...,1,1,Total other comprehensive income that will not...
110,"total other comprehensive income, before tax",1,1,"Total other comprehensive income, before tax"


In [18]:
cashflow_label_summary = (
    cashflow_test_df
    .groupby(
        "normalized_label"
    )
    .agg(
        occurrences=(
            "normalized_label",
            "size"
        ),
        unique_tickers=(
            "ticker",
            "nunique"
        ),
        example_label=(
            "source_label",
            "first"
        )
    )
    .reset_index()
    .sort_values(
        [
            "unique_tickers",
            "occurrences"
        ],
        ascending=[
            False,
            False
        ]
    )
)

display(
    cashflow_label_summary.head(200)
)

,normalized_label,occurrences,unique_tickers,example_label
5,cadangan lindung nilai arus kas,9,3,Cadangan lindung nilai arus kas
48,reserve of cash flow hedges,9,3,Reserve of cash flow hedges
28,laporan arus kas,6,3,Laporan arus kas
49,statement of cash flows,6,3,Statement of cash flows
0,"[1510000] statement of cash flows, direct meth...",3,3,"[1510000] Statement of cash flows, direct meth..."
1,arus kas dari aktivitas investasi,3,3,Arus kas dari aktivitas investasi
2,arus kas dari aktivitas operasi,3,3,Arus kas dari aktivitas operasi
3,arus kas dari aktivitas pendanaan,3,3,Arus kas dari aktivitas pendanaan
4,arus kas sebelum perubahan dalam aset dan liab...,3,3,Arus kas sebelum perubahan dalam aset dan liab...
6,"cash and cash equivalents cash flows, beginnin...",3,3,"Cash and cash equivalents cash flows, beginnin..."


In [19]:
TARGET_PROFIT_PATTERNS = (
    r"gross profit"
    r"|laba bruto"
    r"|laba kotor"
    r"|operating profit"
    r"|profit from operations"
    r"|operating income"
    r"|laba usaha"
    r"|laba operasi"
    r"|profit for the period"
    r"|profit for the year"
    r"|net income"
    r"|net profit"
    r"|laba periode berjalan"
    r"|laba tahun berjalan"
    r"|laba bersih"
)

target_profit_test_df = (
    test_labels_df[
        test_labels_df["normalized_label"]
        .str.contains(
            TARGET_PROFIT_PATTERNS,
            regex=True,
            na=False
        )
    ]
    .copy()
)

print(
    "Target profit labels found:",
    len(target_profit_test_df)
)

display(
    target_profit_test_df[
        [
            "ticker",
            "source_sheet",
            "row_number",
            "source_label",
            "normalized_label"
        ]
    ].head(300)
)

Target profit labels found: 7


,ticker,source_sheet,row_number,source_label,normalized_label
554,ZYRX,1321000,8,Jumlah laba bruto,jumlah laba bruto
555,ZYRX,1321000,8,Total gross profit,total gross profit
8731,ZONE,1311000,8,Jumlah laba bruto,jumlah laba bruto
8732,ZONE,1311000,8,Total gross profit,total gross profit
9522,ZONE,1610000,26,Laba per saham dihitung dengan membagi laba ta...,laba per saham dihitung dengan membagi laba ta...
16887,ZINC,1311000,8,Jumlah laba bruto,jumlah laba bruto
16888,ZINC,1311000,8,Total gross profit,total gross profit


In [20]:
target_profit_summary = (
    target_profit_test_df
    .groupby("normalized_label")
    .agg(
        occurrences=(
            "normalized_label",
            "size"
        ),
        unique_tickers=(
            "ticker",
            "nunique"
        ),
        example_label=(
            "source_label",
            "first"
        )
    )
    .reset_index()
    .sort_values(
        [
            "unique_tickers",
            "occurrences"
        ],
        ascending=[
            False,
            False
        ]
    )
)

display(
    target_profit_summary.head(300)
)

,normalized_label,occurrences,unique_tickers,example_label
0,jumlah laba bruto,3,3,Jumlah laba bruto
2,total gross profit,3,3,Total gross profit
1,laba per saham dihitung dengan membagi laba ta...,1,1,Laba per saham dihitung dengan membagi laba ta...


In [21]:
TARGET_OCF_PATTERNS = (
    r"net cash flows received from \(used in\) operating activities"
    r"|total net cash flows received from \(used in\) operating activities"
    r"|jumlah arus kas bersih yang diperoleh dari \(digunakan untuk\) aktivitas operasi"
)

target_ocf_test_df = (
    test_labels_df[
        test_labels_df["normalized_label"]
        .str.contains(
            TARGET_OCF_PATTERNS,
            regex=True,
            na=False
        )
    ]
    .copy()
)

display(
    target_ocf_test_df[
        [
            "ticker",
            "source_sheet",
            "row_number",
            "source_label",
            "normalized_label"
        ]
    ]
)

,ticker,source_sheet,row_number,source_label,normalized_label
1006,ZYRX,1510000,39,Net cash flows received from (used in) operati...,net cash flows received from (used in) operati...
1021,ZYRX,1510000,47,Jumlah arus kas bersih yang diperoleh dari (di...,jumlah arus kas bersih yang diperoleh dari (di...
1022,ZYRX,1510000,47,Total net cash flows received from (used in) o...,total net cash flows received from (used in) o...
9179,ZONE,1510000,39,Net cash flows received from (used in) operati...,net cash flows received from (used in) operati...
9194,ZONE,1510000,47,Jumlah arus kas bersih yang diperoleh dari (di...,jumlah arus kas bersih yang diperoleh dari (di...
9195,ZONE,1510000,47,Total net cash flows received from (used in) o...,total net cash flows received from (used in) o...
17335,ZINC,1510000,39,Net cash flows received from (used in) operati...,net cash flows received from (used in) operati...
17350,ZINC,1510000,47,Jumlah arus kas bersih yang diperoleh dari (di...,jumlah arus kas bersih yang diperoleh dari (di...
17351,ZINC,1510000,47,Total net cash flows received from (used in) o...,total net cash flows received from (used in) o...


In [22]:
from collections import defaultdict

label_stats = {}

total_files = len(valid_files_df)
CHECKPOINT_EVERY = 250

for i, (_, row) in enumerate(
    tqdm(
        valid_files_df.iterrows(),
        total=total_files,
        desc="Discovering labels",
        unit="file"
    ),
    start=1
):

    file_results = extract_labels_from_file(row)

    seen_labels_in_file = set()
    seen_labels_in_ticker = set()

    ticker = row["ticker"]
    source_file = row["file_name"]

    for item in file_results:

        label = item["normalized_label"]

        if label not in label_stats:
            label_stats[label] = {
                "occurrences": 0,
                "files": set(),
                "tickers": set(),
                "example_label": item["source_label"]
            }

        label_stats[label]["occurrences"] += 1
        label_stats[label]["files"].add(source_file)
        label_stats[label]["tickers"].add(ticker)

    if i % CHECKPOINT_EVERY == 0:

        checkpoint_rows = []

        for label, stats in label_stats.items():

            checkpoint_rows.append({
                "normalized_label": label,
                "occurrences": stats["occurrences"],
                "unique_files": len(stats["files"]),
                "unique_tickers": len(stats["tickers"]),
                "example_label": stats["example_label"]
            })

        checkpoint_df = pd.DataFrame(checkpoint_rows)

        checkpoint_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        tqdm.write(
            f"Checkpoint saved: "
            f"{i:,}/{total_files:,} files | "
            f"{len(checkpoint_df):,} unique labels"
        )

Discovering labels:   2%|▏         | 250/15821 [03:50<2:59:49,  1.44file/s]

Checkpoint saved: 250/15,821 files | 9,776 unique labels


Discovering labels:   3%|▎         | 500/15821 [06:35<3:36:02,  1.18file/s]

Checkpoint saved: 500/15,821 files | 16,173 unique labels


Discovering labels:   5%|▍         | 750/15821 [08:52<2:42:50,  1.54file/s]

...

Discovering labels:  98%|█████████▊| 15500/15821 [6:10:09<19:04,  3.57s/file]

Checkpoint saved: 15,500/15,821 files | 91,839 unique labels


Discovering labels: 100%|█████████▉| 15750/15821 [6:13:26<03:55,  3.31s/file]

Checkpoint saved: 15,750/15,821 files | 91,843 unique labels


Discovering labels: 100%|██████████| 15821/15821 [6:14:16<00:00,  1.42s/file]


In [23]:
label_summary_rows = []

for label, stats in label_stats.items():

    label_summary_rows.append({
        "normalized_label": label,
        "occurrences": stats["occurrences"],
        "unique_files": len(stats["files"]),
        "unique_tickers": len(stats["tickers"]),
        "example_label": stats["example_label"]
    })

label_summary_df = pd.DataFrame(
    label_summary_rows
)

label_summary_df = (
    label_summary_df
    .sort_values(
        [
            "unique_tickers",
            "occurrences"
        ],
        ascending=[
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

display(
    label_summary_df.head(300)
)

,normalized_label,occurrences,unique_files,unique_tickers,example_label
0,aset tetap,135618,15817,948,Aset tetap
1,goodwill,47736,15817,948,Goodwill
2,ekuitas,47436,15816,948,Ekuitas
3,equity,47436,15816,948,Equity
4,saham biasa,47436,15816,948,Saham biasa
...,...,...,...,...,...
295,"bangunan dan fasilitasnya, dimiliki langsung",38696,8040,928,"Bangunan dan fasilitasnya, dimiliki langsung"
296,"building and leasehold improvement, directly o...",38696,8040,928,"Building and leasehold improvement, directly o..."
297,"mesin dan peralatan, dimiliki langsung",38696,8040,928,"Mesin dan peralatan, dimiliki langsung"
298,"machinery and equipment, directly owned",38696,8040,928,"Machinery and equipment, directly owned"


In [24]:
OUTPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_label_discovery.csv"
)

label_summary_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved to:",
    OUTPUT_FILE
)

print(
    "Unique labels:",
    len(label_summary_df)
)

Saved to: data\idx_financial_label_discovery.csv
Unique labels: 91843


In [25]:
profit_labels = label_summary_df[
    label_summary_df["normalized_label"]
    .str.contains(
        r"profit|laba|income",
        regex=True,
        na=False
    )
]

display(
    profit_labels.head(500)
)

,normalized_label,occurrences,unique_files,unique_tickers,example_label
29,saldo laba yang belum ditentukan penggunaannya,47436,15816,948,Saldo laba yang belum ditentukan penggunaannya
36,laba (rugi),31623,15810,948,Laba (rugi)
37,profit (loss),31623,15810,948,Profit (loss)
51,other comprehensive income,31619,15810,948,Other comprehensive income
124,laporan laba rugi dan penghasilan komprehensif...,20624,15816,948,Laporan laba rugi dan penghasilan komprehensif...
...,...,...,...,...,...
6878,beban pajak kini ditentukan berdasarkan laba k...,9,9,2,Beban pajak kini ditentukan berdasarkan laba\n...
6886,beban pajak penghasilan merupakan jumlah pajak...,9,9,2,Beban pajak penghasilan merupakan jumlah pajak...
6887,laba per saham dasar dihitung dengan membagi j...,9,9,2,Laba per saham dasar dihitung dengan membagi j...
6897,laba per saham dihitung dengan membagi laba be...,9,9,2,Laba per saham dihitung dengan membagi laba be...


In [26]:
cashflow_labels = label_summary_df[
    label_summary_df["normalized_label"]
    .str.contains(
        r"cash flow|arus kas|operating activit|aktivitas operasi",
        regex=True,
        na=False
    )
]

display(
    cashflow_labels.head(500)
)

,normalized_label,occurrences,unique_files,unique_tickers,example_label
23,cadangan lindung nilai arus kas,47436,15816,948,Cadangan lindung nilai arus kas
24,reserve of cash flow hedges,47436,15816,948,Reserve of cash flow hedges
128,laporan arus kas,20592,15804,948,Laporan arus kas
129,statement of cash flows,20592,15804,948,Statement of cash flows
200,arus kas dari aktivitas operasi,15807,15804,948,Arus kas dari aktivitas operasi
...,...,...,...,...,...
16417,"pada setiap akhir periode pelaporan, grup meni...",7,7,1,"Pada setiap akhir periode pelaporan, Grup meni..."
16443,"pada setiap akhir periode pelaporan, grup mene...",7,4,1,"Pada setiap akhir periode pelaporan, Grup mene..."
16620,provisi diakui jika grup memiliki kewajiban ki...,6,6,1,Provisi diakui jika Grup memiliki kewajiban ki...
16625,"pada setiap akhir periode pelaporan, perusahaa...",6,6,1,"Pada setiap akhir periode pelaporan, Perusahaa..."
